# 01 — Data Extraction & Source Inspection

**Project:** Olist Customer Churn Analysis  
**Capstone:** NST DVA Capstone 2 — Section D, Group 9  
**Notebook Role:** ETL Stage 1 — Load raw datasets, inspect structure, audit quality, and confirm fitness for cleaning pipeline.

This notebook does **not** modify any raw data. Its sole purpose is to load each of the 7 Olist CSV files, document their structure and quality issues, and produce a written audit that the cleaning notebook (02_cleaning.ipynb) will act on.

All raw files live in `data/raw/` and must never be edited.


## 1. Imports & Project Path Setup

We use `pathlib.Path` for all file references so the notebook runs correctly whether executed locally or on Google Colab. `PROJECT_ROOT` is resolved relative to the notebook's location.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root whether notebook is run from /notebooks/ or from root
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {RAW_DIR}")
print(f"Raw dir exists: {RAW_DIR.exists()}")


Project root : /Users/gokulvks/Documents/DVA_Tablue/SectionD_Group-9_Customer_Churn_Analysis
Raw data dir : /Users/gokulvks/Documents/DVA_Tablue/SectionD_Group-9_Customer_Churn_Analysis/data/raw
Raw dir exists: True


## 2. Raw File Path Definitions

All 7 raw CSV paths are defined as constants here. No path is hardcoded anywhere deeper in the notebook. If a file is missing, the notebook will fail loudly at this cell rather than silently later.


In [2]:
PATH_ORDERS      = RAW_DIR / "olist_orders_dataset.csv"
PATH_CUSTOMERS   = RAW_DIR / "olist_customers_dataset.csv"
PATH_ITEMS       = RAW_DIR / "olist_order_items_dataset.csv"
PATH_PAYMENTS    = RAW_DIR / "olist_order_payments_dataset.csv"
PATH_REVIEWS     = RAW_DIR / "olist_order_reviews_dataset.csv"
PATH_PRODUCTS    = RAW_DIR / "olist_products_dataset.csv"
PATH_CATEGORY    = RAW_DIR / "product_category_name_translation.csv"

ALL_PATHS = {
    "orders"    : PATH_ORDERS,
    "customers" : PATH_CUSTOMERS,
    "items"     : PATH_ITEMS,
    "payments"  : PATH_PAYMENTS,
    "reviews"   : PATH_REVIEWS,
    "products"  : PATH_PRODUCTS,
    "category"  : PATH_CATEGORY,
}

print("File existence check:")
for name, path in ALL_PATHS.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {name}: {path.name}")


File existence check:
  [OK] orders: olist_orders_dataset.csv
  [OK] customers: olist_customers_dataset.csv
  [OK] items: olist_order_items_dataset.csv
  [OK] payments: olist_order_payments_dataset.csv
  [OK] reviews: olist_order_reviews_dataset.csv
  [OK] products: olist_products_dataset.csv
  [OK] category: product_category_name_translation.csv


## 3. Load All Raw Datasets

Each CSV is loaded into its own named DataFrame. No merging, no transformation. We load them individually to isolate issues per file.

**Olist Dataset Context:**  
Olist is a Brazilian marketplace that connects sellers to customers via a single contract. A customer places an order through Olist, which is fulfilled by one or more sellers. These 7 tables together represent the complete order lifecycle — from customer identity to payment, delivery, and review.

| DataFrame | One row represents |
|---|---|
| `df_orders` | One order placed by a customer |
| `df_customers` | One customer entry (note: one real customer can have multiple entries via different `customer_id` values) |
| `df_items` | One item within an order (orders can have multiple items) |
| `df_payments` | One payment installment for an order |
| `df_reviews` | One review submitted for an order |
| `df_products` | One product listed on the platform |
| `df_category` | One Portuguese-to-English category name translation |


In [3]:
df_orders    = pd.read_csv(PATH_ORDERS)
df_customers = pd.read_csv(PATH_CUSTOMERS)
df_items     = pd.read_csv(PATH_ITEMS)
df_payments  = pd.read_csv(PATH_PAYMENTS)
df_reviews   = pd.read_csv(PATH_REVIEWS)
df_products  = pd.read_csv(PATH_PRODUCTS)
df_category  = pd.read_csv(PATH_CATEGORY)

dataframes = {
    "orders"    : df_orders,
    "customers" : df_customers,
    "items"     : df_items,
    "payments"  : df_payments,
    "reviews"   : df_reviews,
    "products"  : df_products,
    "category"  : df_category,
}

print("Load summary:")
for name, df in dataframes.items():
    print(f"  {name:12s} → {df.shape[0]:>7,} rows × {df.shape[1]} cols")


Load summary:
  orders       →  99,441 rows × 8 cols
  customers    →  99,441 rows × 5 cols
  items        → 112,650 rows × 7 cols
  payments     → 103,886 rows × 5 cols
  reviews      →  99,224 rows × 7 cols
  products     →  32,951 rows × 9 cols
  category     →      71 rows × 2 cols


## 4. Schema Inspection — dtypes and Column Names

For each table we inspect column names and data types. This is the foundation for identifying type casting work needed in 02_cleaning.ipynb — particularly all timestamp columns which are stored as `object` (string) and must be converted to `datetime64`.


In [4]:
for name, df in dataframes.items():
    print(f"\n{'='*55}")
    print(f"  TABLE: {name.upper()}")
    print(f"{'='*55}")
    print(df.dtypes.to_string())



  TABLE: ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

  TABLE: CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

  TABLE: ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

  TABLE: PAYMENTS
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64

  TABLE: REVIEWS
review_id                    str
order_id                     str
r

## 5. Head Preview — First 3 Rows of Each Table

A quick preview to visually confirm the data loaded correctly and values look sensible.


In [5]:
for name, df in dataframes.items():
    print(f"\n{'='*55}")
    print(f"  TABLE: {name.upper()} — first 3 rows")
    print(f"{'='*55}")
    display(df.head(3))



  TABLE: ORDERS — first 3 rows


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



  TABLE: CUSTOMERS — first 3 rows


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



  TABLE: ITEMS — first 3 rows


...

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24



  TABLE: PRODUCTS — first 3 rows


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0



  TABLE: CATEGORY — first 3 rows


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 6. Null Audit — Missing Values Per Column

We count nulls in every column across all 7 tables. Only columns with at least one null are shown. This audit directly informs the null-handling strategy in 02_cleaning.ipynb.


In [6]:
print("NULL AUDIT — columns with missing values:\n")
for name, df in dataframes.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"  TABLE: {name.upper()}")
    if nulls.empty:
        print("    → No nulls found.\n")
    else:
        for col, count in nulls.items():
            pct = count / len(df) * 100
            print(f"    {col:<40s} {count:>6,} nulls  ({pct:.1f}%)")
        print()


NULL AUDIT — columns with missing values:

  TABLE: ORDERS
    order_approved_at                           160 nulls  (0.2%)
    order_delivered_carrier_date              1,783 nulls  (1.8%)
    order_delivered_customer_date             2,965 nulls  (3.0%)

  TABLE: CUSTOMERS
    → No nulls found.

  TABLE: ITEMS
    → No nulls found.

  TABLE: PAYMENTS
    → No nulls found.

  TABLE: REVIEWS
    review_comment_title                     87,656 nulls  (88.3%)
    review_comment_message                   58,247 nulls  (58.7%)

  TABLE: PRODUCTS
    product_category_name                       610 nulls  (1.9%)
    product_name_lenght                         610 nulls  (1.9%)
    product_description_lenght                  610 nulls  (1.9%)
    product_photos_qty                          610 nulls  (1.9%)
    product_weight_g                              2 nulls  (0.0%)
    product_length_cm                             2 nulls  (0.0%)
    product_height_cm                             2 nul

### Null Audit Findings — Interpretation

**df_orders:**
- `order_approved_at` — 160 nulls (~0.2%). Orders where approval was not recorded. Will be filled with `order_purchase_timestamp` in cleaning since approval typically follows purchase immediately for these cases.
- `order_delivered_carrier_date` — 1,783 nulls (~1.8%). Orders not yet picked up by carrier at data capture time. After filtering to `status = delivered`, residual nulls will be dropped.
- `order_delivered_customer_date` — 2,965 nulls (~3.0%). Orders not yet delivered at data capture. After filtering to delivered status, any remaining nulls here indicate data recording gaps and will be dropped as they cannot be used to compute delivery delay — a core analytical column.

**df_reviews:**
- `review_comment_title` — 87,656 nulls (~88.3%). This is expected. Review comments are optional on the Olist platform. The null rate here is not a data quality problem. These will be filled with an empty string in cleaning.
- `review_comment_message` — 58,247 nulls (~58.7%). Same reasoning. `review_score` (the 1–5 star rating) has zero nulls and is the only review column used analytically.

**df_products:**
- `product_category_name` — 610 nulls (~1.9%). Products without a category label. Will be filled with `'unknown'` in cleaning.
- Physical dimension columns (`product_name_lenght`, `product_description_lenght`, `product_photos_qty`) — 610 nulls each, same products. Will be handled alongside the category null.
- `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` — 2 nulls each. These 2 rows will be dropped in cleaning.

**df_customers, df_items, df_payments, df_category:**
- No nulls. These tables are clean on entry.


## 7. Duplicate Audit — Exact Row Duplicates and Primary Key Uniqueness

We check two things:
1. Exact duplicate rows in each table (all columns identical).
2. Uniqueness of the primary key in each table, because downstream merges depend on key integrity.

**Important note on `df_customers`:** The column `customer_id` is NOT a unique customer identifier. One real customer can place multiple orders and will appear in this table multiple times with different `customer_id` values, but the same `customer_unique_id`. The churn label in 02_cleaning.ipynb must therefore be built on `customer_unique_id`, not `customer_id`. This distinction is critical.


In [7]:
print("DUPLICATE AUDIT:\n")

# Exact row duplicates
for name, df in dataframes.items():
    dup_count = df.duplicated().sum()
    print(f"  {name:12s} — exact row duplicates: {dup_count:,}")

print()

# Primary key uniqueness checks
pk_checks = [
    ("orders",    df_orders,    "order_id"),
    ("customers", df_customers, "customer_id"),
    ("customers", df_customers, "customer_unique_id"),
    ("items",     df_items,     ["order_id", "order_item_id"]),
    ("payments",  df_payments,  ["order_id", "payment_sequential"]),
    ("reviews",   df_reviews,   "review_id"),
    ("products",  df_products,  "product_id"),
    ("category",  df_category,  "product_category_name"),
]

print("PRIMARY KEY UNIQUENESS:\n")
for table_name, df, key in pk_checks:
    if isinstance(key, list):
        dup = df.duplicated(subset=key).sum()
        key_label = " + ".join(key)
    else:
        dup = df[key].duplicated().sum()
        key_label = key
    status = "UNIQUE" if dup == 0 else f"NOT UNIQUE — {dup:,} duplicates"
    print(f"  {table_name:12s} [{key_label}]: {status}")


DUPLICATE AUDIT:

  orders       — exact row duplicates: 0
  customers    — exact row duplicates: 0
  items        — exact row duplicates: 0
  payments     — exact row duplicates: 0
  reviews      — exact row duplicates: 0
  products     — exact row duplicates: 0
  category     — exact row duplicates: 0

PRIMARY KEY UNIQUENESS:

  orders       [order_id]: UNIQUE
  customers    [customer_id]: UNIQUE
  customers    [customer_unique_id]: NOT UNIQUE — 3,345 duplicates
  items        [order_id + order_item_id]: UNIQUE
  payments     [order_id + payment_sequential]: UNIQUE
  reviews      [review_id]: NOT UNIQUE — 814 duplicates
  products     [product_id]: UNIQUE
  category     [product_category_name]: UNIQUE


## 8. Order Status Distribution

The `order_status` column in `df_orders` determines which rows are valid for churn analysis. We inspect the full distribution here to document the cleaning decision made in 02_cleaning.ipynb.

**Decision documented here:** Only orders with `order_status == 'delivered'` will be retained for analysis. Non-delivered statuses (canceled, shipped, processing, etc.) do not represent completed customer transactions and cannot be used to derive churn behavior reliably. Any customer whose only interaction was a canceled or pending order would be incorrectly labeled as churned if these rows were included.


In [8]:
status_counts = df_orders["order_status"].value_counts()
status_pct    = df_orders["order_status"].value_counts(normalize=True) * 100

print("ORDER STATUS DISTRIBUTION:\n")
print(f"  {'Status':<20s} {'Count':>8}   {'Pct':>6}")
print(f"  {'-'*20}   {'-'*8}   {'-'*6}")
for status in status_counts.index:
    print(f"  {status:<20s} {status_counts[status]:>8,}   {status_pct[status]:>5.1f}%")

print(f"\n  Total rows: {len(df_orders):,}")
delivered_count = status_counts.get("delivered", 0)
non_delivered   = len(df_orders) - delivered_count
print(f"  Rows retained after filter (delivered): {delivered_count:,}")
print(f"  Rows to be dropped (non-delivered):     {non_delivered:,}")


ORDER STATUS DISTRIBUTION:

  Status                  Count      Pct
  --------------------   --------   ------
  delivered              96,478    97.0%
  shipped                 1,107     1.1%
  canceled                  625     0.6%
  unavailable               609     0.6%
  invoiced                  314     0.3%
  processing                301     0.3%
  created                     5     0.0%
  approved                    2     0.0%

  Total rows: 99,441
  Rows retained after filter (delivered): 96,478
  Rows to be dropped (non-delivered):     2,963


## 9. Customer Identity — `customer_id` vs `customer_unique_id`

This is the most important structural insight in the extraction phase for a churn project.

On Olist, when a customer places a new order, a new `customer_id` is generated for that order — even if the same real person already placed an order before. The `customer_unique_id` is the stable identifier that tracks the same real customer across multiple orders.

If we built the churn label using `customer_id`, every customer would appear to have placed only one order (because each order gets a fresh `customer_id`), and the churn rate would be artificially 100%. The correct approach — which will be implemented in 02_cleaning.ipynb — is to use `customer_unique_id` to count orders per real customer.

We verify this below.


In [9]:
total_customer_ids        = df_customers["customer_id"].nunique()
total_unique_customer_ids = df_customers["customer_unique_id"].nunique()

print("CUSTOMER IDENTITY AUDIT:\n")
print(f"  Unique customer_id values        : {total_customer_ids:,}")
print(f"  Unique customer_unique_id values : {total_unique_customer_ids:,}")
print(f"  Difference                       : {total_customer_ids - total_unique_customer_ids:,}")
print()

# Find customers who placed more than one order
orders_per_unique = (
    df_orders
    .merge(df_customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

repeat_customers = (orders_per_unique > 1).sum()
total_unique     = orders_per_unique.shape[0]
repeat_pct       = repeat_customers / total_unique * 100

print(f"  Real customers (by customer_unique_id): {total_unique:,}")
print(f"  Repeat customers (2+ orders)           : {repeat_customers:,}  ({repeat_pct:.2f}%)")
print(f"  One-time customers (1 order)           : {total_unique - repeat_customers:,}  ({100 - repeat_pct:.2f}%)")
print()
print("  --> The one-time customer % above is the expected approximate churn rate.")
print("      This will be confirmed with a formal churn label in 02_cleaning.ipynb.")


CUSTOMER IDENTITY AUDIT:

  Unique customer_id values        : 99,441
  Unique customer_unique_id values : 96,096
  Difference                       : 3,345

  Real customers (by customer_unique_id): 96,096
  Repeat customers (2+ orders)           : 2,997  (3.12%)
  One-time customers (1 order)           : 93,099  (96.88%)

  --> The one-time customer % above is the expected approximate churn rate.
      This will be confirmed with a formal churn label in 02_cleaning.ipynb.


## 10. Column Name Quality Check

We flag any column names that have known typos or inconsistencies. The `olist_products_dataset.csv` has two columns with spelling errors (`lenght` instead of `length`). These will be corrected with `rename()` in 02_cleaning.ipynb.


In [10]:
print("ALL COLUMN NAMES BY TABLE:\n")
for name, df in dataframes.items():
    print(f"  {name.upper()}:")
    for col in df.columns:
        flag = " ← TYPO (will rename in NB02)" if "lenght" in col else ""
        print(f"    {col}{flag}")
    print()


ALL COLUMN NAMES BY TABLE:

  ORDERS:
    order_id
    customer_id
    order_status
    order_purchase_timestamp
    order_approved_at
    order_delivered_carrier_date
    order_delivered_customer_date
    order_estimated_delivery_date

  CUSTOMERS:
    customer_id
    customer_unique_id
    customer_zip_code_prefix
    customer_city
    customer_state

  ITEMS:
    order_id
    order_item_id
    product_id
    seller_id
    shipping_limit_date
    price
    freight_value

  PAYMENTS:
    order_id
    payment_sequential
    payment_type
    payment_installments
    payment_value

  REVIEWS:
    review_id
    order_id
    review_score
    review_comment_title
    review_comment_message
    review_creation_date
    review_answer_timestamp

  PRODUCTS:
    product_id
    product_category_name
    product_name_lenght ← TYPO (will rename in NB02)
    product_description_lenght ← TYPO (will rename in NB02)
    product_photos_qty
    product_weight_g
    product_length_cm
    product_height_c

## 11. Join Key Audit — Verifying Merge Columns

Before merging in 02_cleaning.ipynb, we verify that the join keys exist in the correct tables and that foreign key values are not introducing unexpected nulls after a left join. We sample a few key overlaps here.


In [11]:
print("JOIN KEY AUDIT:\n")

# order_id coverage: items, payments, reviews vs orders
for name, df in [("items", df_items), ("payments", df_payments), ("reviews", df_reviews)]:
    order_ids_in_orders = set(df_orders["order_id"])
    order_ids_in_child  = set(df[df.columns[df.columns.str.contains("order_id")][0]])
    in_both             = order_ids_in_child & order_ids_in_orders
    only_in_child       = order_ids_in_child - order_ids_in_orders
    print(f"  {name:10s} order_id → orders order_id:")
    print(f"    Matched   : {len(in_both):,}")
    print(f"    Only in {name:8s}: {len(only_in_child):,}  (orphaned, will be lost on left join from orders)")
    print()

# product_id coverage: items vs products
product_ids_in_products = set(df_products["product_id"])
product_ids_in_items    = set(df_items["product_id"])
unmatched_in_items      = product_ids_in_items - product_ids_in_products
print(f"  items product_id → products product_id:")
print(f"    Matched   : {len(product_ids_in_items & product_ids_in_products):,}")
print(f"    In items but not in products: {len(unmatched_in_items):,}")
print()

# category translation coverage
categories_in_products   = set(df_products["product_category_name"].dropna())
categories_in_translation = set(df_category["product_category_name"])
untranslated             = categories_in_products - categories_in_translation
print(f"  products category_name → category translation:")
print(f"    Matched         : {len(categories_in_products & categories_in_translation):,}")
print(f"    Not in translation table: {len(untranslated):,}")
if untranslated:
    print(f"    Examples: {list(untranslated)[:5]}")


JOIN KEY AUDIT:

  items      order_id → orders order_id:
    Matched   : 98,666
    Only in items   : 0  (orphaned, will be lost on left join from orders)

  payments   order_id → orders order_id:
    Matched   : 99,440
    Only in payments: 0  (orphaned, will be lost on left join from orders)

  reviews    order_id → orders order_id:
    Matched   : 98,673
    Only in reviews : 0  (orphaned, will be lost on left join from orders)

  items product_id → products product_id:
    Matched   : 32,951
    In items but not in products: 0

  products category_name → category translation:
    Matched         : 71
    Not in translation table: 2
    Examples: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


## 12. Schema Summary Table

A consolidated reference table documenting all 7 raw files. This table will be reproduced in `docs/data_dictionary.md` and referenced in the final presentation (Slide 3 — Data Engineering).


In [12]:
schema_summary = [
    {
        "Table"       : "olist_orders_dataset.csv",
        "Rows"        : f"{len(df_orders):,}",
        "Cols"        : len(df_orders.columns),
        "Primary Key" : "order_id",
        "Joins To"    : "customers (customer_id)",
        "Key Nulls"   : "order_approved_at (160), order_delivered_customer_date (2,965)",
    },
    {
        "Table"       : "olist_customers_dataset.csv",
        "Rows"        : f"{len(df_customers):,}",
        "Cols"        : len(df_customers.columns),
        "Primary Key" : "customer_id",
        "Joins To"    : "orders (customer_id)",
        "Key Nulls"   : "None",
    },
    {
        "Table"       : "olist_order_items_dataset.csv",
        "Rows"        : f"{len(df_items):,}",
        "Cols"        : len(df_items.columns),
        "Primary Key" : "order_id + order_item_id",
        "Joins To"    : "orders (order_id), products (product_id)",
        "Key Nulls"   : "None",
    },
    {
        "Table"       : "olist_order_payments_dataset.csv",
        "Rows"        : f"{len(df_payments):,}",
        "Cols"        : len(df_payments.columns),
        "Primary Key" : "order_id + payment_sequential",
        "Joins To"    : "orders (order_id)",
        "Key Nulls"   : "None",
    },
    {
        "Table"       : "olist_order_reviews_dataset.csv",
        "Rows"        : f"{len(df_reviews):,}",
        "Cols"        : len(df_reviews.columns),
        "Primary Key" : "review_id",
        "Joins To"    : "orders (order_id)",
        "Key Nulls"   : "review_comment_title (87,656), review_comment_message (58,247)",
    },
    {
        "Table"       : "olist_products_dataset.csv",
        "Rows"        : f"{len(df_products):,}",
        "Cols"        : len(df_products.columns),
        "Primary Key" : "product_id",
        "Joins To"    : "items (product_id), category (product_category_name)",
        "Key Nulls"   : "product_category_name (610), dimension cols (610 or 2)",
    },
    {
        "Table"       : "product_category_name_translation.csv",
        "Rows"        : f"{len(df_category):,}",
        "Cols"        : len(df_category.columns),
        "Primary Key" : "product_category_name",
        "Joins To"    : "products (product_category_name)",
        "Key Nulls"   : "None",
    },
]

df_schema = pd.DataFrame(schema_summary)
print("SCHEMA SUMMARY:\n")
print(df_schema.to_string(index=False))


SCHEMA SUMMARY:

                                Table    Rows  Cols                   Primary Key                                             Joins To                                                      Key Nulls
             olist_orders_dataset.csv  99,441     8                      order_id                              customers (customer_id) order_approved_at (160), order_delivered_customer_date (2,965)
          olist_customers_dataset.csv  99,441     5                   customer_id                                 orders (customer_id)                                                           None
        olist_order_items_dataset.csv 112,650     7      order_id + order_item_id             orders (order_id), products (product_id)                                                           None
     olist_order_payments_dataset.csv 103,886     5 order_id + payment_sequential                                    orders (order_id)                                                         

## 13. Extraction Audit Summary — Issues to Address in 02_cleaning.ipynb

This is the definitive handoff document from NB01 to NB02. Every item listed here has a corresponding action in the cleaning notebook.

| # | Issue | Location | Action in NB02 |
|---|---|---|---|
| 1 | All 5 timestamp columns stored as `object` (string) | `df_orders` | `pd.to_datetime(..., errors='coerce')` |
| 2 | `order_approved_at` — 160 nulls | `df_orders` | Fill with `order_purchase_timestamp` |
| 3 | `order_delivered_carrier_date` — 1,783 nulls | `df_orders` | Drop after filtering to delivered |
| 4 | `order_delivered_customer_date` — 2,965 nulls | `df_orders` | Drop rows after filtering to delivered |
| 5 | Non-delivered orders (2,963 rows) | `df_orders` | Filter to `order_status == 'delivered'` only |
| 6 | `review_comment_title` — 87,656 nulls | `df_reviews` | Fill with `'no_comment'` (expected, optional field) |
| 7 | `review_comment_message` — 58,247 nulls | `df_reviews` | Fill with `'no_comment'` (expected, optional field) |
| 8 | `product_category_name` — 610 nulls | `df_products` | Fill with `'unknown'` |
| 9 | Dimension cols — 2 nulls | `df_products` | Drop those 2 rows |
| 10 | Typo in `product_name_lenght` | `df_products` | Rename to `product_name_length` |
| 11 | Typo in `product_description_lenght` | `df_products` | Rename to `product_description_length` |
| 12 | `customer_id` ≠ unique customer | `df_customers` / `df_orders` | Use `customer_unique_id` for churn label |
| 13 | `shipping_limit_date` stored as string | `df_items` | `pd.to_datetime(..., errors='coerce')` |
| 14 | `review_creation_date`, `review_answer_timestamp` stored as string | `df_reviews` | `pd.to_datetime(..., errors='coerce')` |

**Extraction phase complete. No data was modified. All files in `data/raw/` remain untouched.**
